# Extraction dashboard environment

In [ ]:
from pathlib import Path
import subprocess
import sys

on_drive = False

DATASET = 'dataset_name'
L_H_SETTING = '168_24'
MODEL_NAME = 'chronos2'
SPACE, METRIC, K, RETRIEVAL_MODE = 'raw', 'euclidean', 1, 'online'
RUN = 'run_0'
EXTRACTION_RELATIVE_DIR = (
    Path('extraction') / DATASET / L_H_SETTING / MODEL_NAME
    / SPACE / METRIC / str(K) / RETRIEVAL_MODE / RUN
)

COLAB_OUTPUTS_ROOT = Path('/content/drive/MyDrive/path/to/outputs')
COLAB_REPO_DIR = '/content/drive/MyDrive/path/to/ts-ifa'
INSTALL_PROJECT_ON_COLAB = True
INSTALL_WIDGETS_ON_COLAB = True

if on_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path(COLAB_REPO_DIR).expanduser().resolve()
    %cd $PROJECT_ROOT
    if INSTALL_PROJECT_ON_COLAB:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            '--no-deps', '-e', COLAB_REPO_DIR,
        ])
    if INSTALL_WIDGETS_ON_COLAB:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets>=8.1',
        ])
    OUTPUTS_ROOT = COLAB_OUTPUTS_ROOT
else:
    kernel_dir = Path.cwd().resolve()
    PROJECT_ROOT = next((
        candidate
        for candidate in (kernel_dir, *kernel_dir.parents)
        if (candidate / 'src' / 'visu' / 'dashboard.py').is_file()
    ), None)
    if PROJECT_ROOT is None:
        raise RuntimeError(
            'Start Jupyter from the adaptation project root before opening this notebook.'
        )
    OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXTRACTION_DIR = OUTPUTS_ROOT / EXTRACTION_RELATIVE_DIR
EXTRACTION_DIR

# Imports

In [ ]:
from IPython.display import display

from src.visu.dashboard import (
    data_summary,
    horizon_section,
    load_extraction_dashboard_data,
    query_section,
    window_scatter_section,
)

# Data loading

In [ ]:
data = load_extraction_dashboard_data(EXTRACTION_DIR)
print(data_summary(data))

# Query and retrieved examples

For a query origin $s$, the solid segment is the observed lookback and the dashed segment is its future:

$$X_s=(z_{s-L+1},\ldots,z_s),\qquad Y_s=(z_{s+1},\ldots,z_{s+H}).$$

Each retrieved neighbour $r_j$ is drawn in the same way as $(X_{r_j},Y_{r_j})$.

In [ ]:
display(query_section(data))

# Extraction prediction diagnostics

These widgets compare the frozen vanilla forecast with the retrieval-context forecast saved by extraction.

In [ ]:
display(window_scatter_section(data))

In [ ]:
display(horizon_section(data))